In [5]:
import open3d as o3d
import numpy as np

# Load the full multi-view incremental point cloud
pcd = o3d.io.read_point_cloud("../outputs/final_reconstruction.ply")
print(f"Loaded {len(pcd.points)} 3D points for surface meshing.")

Loaded 72627 3D points for surface meshing.


In [6]:
# 1. Estimate unoriented surface normals via Local PCA (k-NN Search)
# Adjust radius based on your point cloud scale (e.g., 0.05 to 0.2 meters)
search_param = o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30)
pcd.estimate_normals(search_param=search_param)

# 2. Propagate consistent normal orientation across adjacent tangent planes
pcd.orient_normals_consistent_tangent_plane(k=15)

# Verify normals were calculated successfully
print(f"Has normals: {pcd.has_normals()}")
print(f"Sample normal vector [0]: {np.asarray(pcd.normals)[0]}")

# 3. Optional: Inspect oriented point cloud with normal vectors
o3d.visualization.draw_geometries([pcd], point_show_normal=True)

Has normals: True
Sample normal vector [0]: [ 0.21631454 -0.61796018 -0.75586588]



(python:44233): Gtk-WARNING **: 16:36:22.542: gtk_disable_setlocale() must be called before gtk_init()


In [7]:
# 1. Run Screened Poisson Surface Reconstruction
# depth controls the octree resolution (depth=8 to 10 is standard)
mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
    pcd, depth=9, scale=1.1
)

print(f"Raw Mesh: {len(mesh.vertices)} vertices | {len(mesh.triangles)} triangles")

# 2. Trim low-density boundary artifacts ("floaters")
# Poisson builds a watertight boundary box. Regions where no 3D points existed
# (like empty sky/edges) get low-density "ghost" triangles that must be cropped.
densities = np.asarray(densities)
density_threshold = np.quantile(densities, 0.05)  # remove bottom 5% lowest density
vertices_to_remove = densities < density_threshold
mesh.remove_vertices_by_mask(vertices_to_remove)

print(f"Trimmed Mesh: {len(mesh.vertices)} vertices | {len(mesh.triangles)} triangles remaining")

# 3. Export the 3D mesh
o3d.io.write_triangle_mesh("../outputs/building_mesh.ply", mesh)

Raw Mesh: 202115 vertices | 405106 triangles
Trimmed Mesh: 192009 vertices | 382197 triangles remaining


True

In [8]:
from pathlib import Path

# 1. Define output directory and file path
output_dir = Path("../outputs")
output_dir.mkdir(parents=True, exist_ok=True)
mesh_path = output_dir / "building_mesh.ply"

# 2. Compute vertex normals for smooth lighting and shading
mesh.compute_vertex_normals()

# 3. Export the trimmed triangle mesh
success = o3d.io.write_triangle_mesh(str(mesh_path), mesh)
if success:
    print(f"Successfully saved 3D mesh ({len(mesh.vertices)} vertices, {len(mesh.triangles)} triangles) to: {mesh_path.resolve()}")

# 4. Render the interactive 3D surface mesh
o3d.visualization.draw_geometries(
    [mesh],
    window_name="Reconstructed 3D Surface Mesh",
    width=1024,
    height=768,
    mesh_show_wireframe=False,
    mesh_show_back_face=True,
)

Successfully saved 3D mesh (192009 vertices, 382197 triangles) to: /home/semi/projects/propyläen_3d_reconstruct/outputs/building_mesh.ply



(python:44233): Gtk-WARNING **: 16:36:28.015: gtk_disable_setlocale() must be called before gtk_init()
